In [1]:
import json
import os, statistics, math

def extract_step_to_acc(path: str):
    """
    Read a raw metrics file and return {step: test_acc} from even-numbered lines only.
    - 1-based line numbering (keep only even lines)
    - JSON parse; keep only split == 'test'
    - Map: step -> acc (latest occurrence wins if duplicated)
    """
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):  # 1-based
            line = line.strip()
            if not line or (i % 2 != 0):  # skip odd lines
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("split") != "test":
                continue
            step = obj.get("step")
            acc = obj.get("acc")
            if step is not None and acc is not None:
                result[step] = acc  # latest one wins
    return result

In [2]:
def summarize_by_step(root='.', start=1, end=53, prefix='client_', ext='.raw',
                      variance='population', step_keys=None, require_all=False):
    """
    Aggregate across clients PER STEP and return TWO dictionaries:
      - mean_by_step: {step: mean_test_acc_over_clients}
      - var_by_step:  {step: variance_test_acc_over_clients}

    Parameters:
      - variance: 'population' (statistics.pvariance) | 'sample' (statistics.variance)
      - step_keys: optional iterable of steps to enforce (e.g., [0,10,20,...,100]).
                   If None, uses the union of steps observed across clients.
      - require_all: if True, only include a step if ALL clients have a value for it.
                     If False, compute from available values.

    Notes:
      - Uses extract_step_to_acc(path) from earlier cell (even-line, split=='test').
      - Missing files or missing steps are ignored per 'require_all' policy.
    """
    # Collect per-step lists of accuracies across clients
    accs_by_step = {}
    client_count = 0
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            d = extract_step_to_acc(path)
        except FileNotFoundError:
            # Missing client file; skip
            continue
        client_count += 1
        for step, acc in d.items():
            accs_by_step.setdefault(step, []).append(acc)

    # Decide which steps to include
    if step_keys is None:
        steps = sorted(accs_by_step.keys())
    else:
        steps = list(step_keys)

    mean_by_step = {}
    var_by_step  = {}
    for step in steps:
        vals = accs_by_step.get(step, [])
        if not vals:
            continue  # no data at this step
        if require_all and len(vals) != client_count:
            # Skip this step because not all clients provided it
            continue
        m = sum(vals) / len(vals)
        if variance == 'population':
            v = statistics.pvariance(vals)
        elif variance == 'sample':
            v = statistics.variance(vals) if len(vals) > 1 else float('nan')
        else:
            raise ValueError("variance must be 'population' or 'sample'")
        mean_by_step[step] = m
        var_by_step[step]  = v
    return mean_by_step, var_by_step

In [3]:
mean_dict, var_dict = summarize_by_step(root='.', start=1, end=53)
print(mean_dict)
print(var_dict)

{0: 0.7889999999999999, 10: 0.7870000000000001, 20: 0.782, 30: 0.7855000000000001, 40: 0.7825, 50: 0.7925000000000002, 60: 0.7910000000000001, 70: 0.796, 80: 0.799, 90: 0.8020000000000002, 100: 0.8049999999999999, 110: 0.8069999999999998, 120: 0.8029999999999999, 130: 0.807, 140: 0.8035, 150: 0.8115, 160: 0.8155000000000001, 170: 0.8109999999999999, 180: 0.8095000000000002, 190: 0.8090000000000002, 200: 0.8150000000000001, 210: 0.8195, 220: 0.8145, 230: 0.8155000000000001, 240: 0.8195000000000002, 250: 0.8215000000000001, 260: 0.8194871794871794, 270: 0.8157894736842107, 280: 0.8278947368421052, 290: 0.8231578947368421, 300: 0.8257894736842104}
{0: 0.021359, 10: 0.022351, 20: 0.020196, 30: 0.02097975, 40: 0.01958375, 50: 0.01861375, 60: 0.020478999999999997, 70: 0.018484, 80: 0.017658999999999998, 90: 0.018116, 100: 0.014895, 110: 0.016651, 120: 0.017870999999999998, 130: 0.013630999999999999, 140: 0.01485775, 150: 0.01471775, 160: 0.014429749999999998, 170: 0.014439, 180: 0.0131197499